# 07 - FD001 temporal features

Objetivo: evaluar si una ventana temporal de 30 ciclos mejora la prediccion de RUL
frente al enfoque current-cycle.

Para cada feature base se calculan: ultimo valor, media, desvio, minimo, maximo,
delta y pendiente lineal.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessed_FD001 import (
    DEFAULT_CUT_RULS,
    DEFAULT_RUL_CAP,
    DEFAULT_TEMPORAL_WINDOW,
    prepare_fd001_temporal_with_cutoff_eval,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_rul_bin,
    plot_validation_diagnostics,
    train_and_predict_models,
)

RESULTS_DIR = PROJECT_ROOT / "results" / "FD001"
FIGURE_DIR = PROJECT_ROOT / "figures" / "fd001_temporal"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## Preparacion temporal

La ventana usa solo informacion disponible hasta el ciclo de prediccion. Para
validacion se usan los mismos cortes artificiales que en el notebook 06.

In [2]:
prepared = prepare_fd001_temporal_with_cutoff_eval(
    max_rul=DEFAULT_RUL_CAP,
    cut_ruls=DEFAULT_CUT_RULS,
    window_size=DEFAULT_TEMPORAL_WINDOW,
    random_state=RANDOM_STATE,
)

display(preprocessing_summary(prepared))
print("Features temporales:", len(prepared["feature_columns"]))
print("Features base:", len(prepared["base_feature_columns"]))
display(prepared["eval_df"][["unit", "cycle", "cut_rul", "RUL_raw", "window_size_used"]].head())

,split,rows,units,features,target_mean,target_min,target_max
0,train,16561,80,119,86.958819,0.0,125.0
1,eval,99,20,119,79.393939,20.0,140.0
2,test_last,100,100,119,75.520000,7.0,145.0


Features temporales: 119
Features base: 17


,unit,cycle,cut_rul,RUL_raw,window_size_used
0,1,52,140,140.0,30
1,1,82,110,110.0,30
2,1,112,80,80.0,30
3,1,142,50,50.0,30
4,1,172,20,20.0,30


## Entrenamiento

Para temporal no repetimos el dummy; el dummy ya quedo como baseline minimo en
current-cycle. Comparamos Ridge, RandomForest y MLP.

In [3]:
metrics, validation_predictions, fitted_models = train_and_predict_models(
    prepared,
    representation=f"temporal_w{DEFAULT_TEMPORAL_WINDOW}",
    include_dummy=False,
    random_state=RANDOM_STATE,
)

metrics_path = RESULTS_DIR / "fd001_temporal_metrics.csv"
predictions_path = RESULTS_DIR / "fd001_temporal_validation_predictions.csv"
bin_metrics_path = RESULTS_DIR / "fd001_temporal_metrics_by_rul_bin.csv"

metrics.to_csv(metrics_path, index=False)
validation_predictions.to_csv(predictions_path, index=False)
metrics_by_rul_bin(validation_predictions).to_csv(bin_metrics_path, index=False)

display(metrics)
print("Guardado:", metrics_path)
print("Guardado:", predictions_path)

,representation,model_name,n_features,target_used_for_training,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct
0,temporal_w30,RandomForestRegressor,119,RUL_capped,99,13.817341,17.912073,0.819898,580.933845,5.868019,14.141414
1,temporal_w30,MLP,119,RUL_capped,99,14.311264,18.734374,0.802983,633.588376,6.399883,15.151515
2,temporal_w30,Ridge,119,RUL_capped,99,18.419013,23.133112,0.699604,1073.335072,10.841768,17.171717


Guardado: C:\Users\frane\udesa\ML\vs\TPF-ML\results\fd001_temporal_metrics.csv
Guardado: C:\Users\frane\udesa\ML\vs\TPF-ML\results\fd001_temporal_validation_predictions.csv


## Graficos de diagnostico

Los graficos se guardan en `figures/fd001_temporal/`.

In [4]:
plot_validation_diagnostics(
    validation_predictions,
    figure_dir=FIGURE_DIR,
    prefix=f"FD001 temporal w={DEFAULT_TEMPORAL_WINDOW}",
)

sorted(FIGURE_DIR.glob("*.png"))

[WindowsPath('C:/Users/frane/udesa/ML/vs/TPF-ML/figures/fd001_temporal/error_vs_true_rul.png'),
 WindowsPath('C:/Users/frane/udesa/ML/vs/TPF-ML/figures/fd001_temporal/mae_by_rul_bin.png'),
 WindowsPath('C:/Users/frane/udesa/ML/vs/TPF-ML/figures/fd001_temporal/predicted_vs_true.png'),
 WindowsPath('C:/Users/frane/udesa/ML/vs/TPF-ML/figures/fd001_temporal/worst_cases_abs_error.png')]

## Lectura rapida

Si temporal mejora, la evidencia favorece usar la evolucion reciente del motor,
no solo el estado puntual del ciclo actual.